## 35. 프로젝트 루트 설정

In [2]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: /Users/chaehuiju/dev/ai-data-analysis
데이터 폴더: /Users/chaehuiju/dev/ai-data-analysis/data/raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [3]:
import pandas as pd

# 데이터프레임 정의
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (766, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [5]:
# 병합 실습 전에 기준 테이블의 주요 키가 고유한지 확인합니다.
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## 38. Series와 DataFrame 선택

In [6]:
city_series = customers["city"]

customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))

display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [7]:
# 하나만 필터링해서 보겠다.
customers_over_30 = customers[
    customers["age"] >= 30
]

print(len(customers), len(customers_over_30))

display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,장춘자,F,32,대구,2023-11-12
2,3,김상현,F,61,성남,2025-02-12
3,4,김재호,F,55,울산,2025-03-28
5,6,한순자,F,32,성남,2024-01-26
6,7,이미숙,F,53,인천,2023-11-23


## 40. 복합 조건 필터링

In [8]:
# 30세 이상이면서 서울 거주:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울") # & 이므로 동시에 조건 만족해야함 
]

display(seoul_over_30.head()) # 5개 뽑아서 보여줌

# 서울 또는 부산:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"]) # isin() city열에서 서울 혹은 부산이 들어간 것
]

display(
    seoul_or_busan["city"].value_counts() # value_count() 중요한 메소드!!! 값의 개수를 count
)

# 완료 주문이 아닌 주문:
not_completed = orders[
    ~(orders["order_status"] == "completed") # ~ 는 여집합
]

display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

,customer_id,name,gender,age,city,signup_date
8,9,우영미,M,69,서울,2025-04-05
14,15,서민준,M,69,서울,2024-09-27
29,30,윤우진,F,32,서울,2024-01-23
47,48,박승현,F,47,서울,2024-06-09
65,66,양영미,F,39,서울,2024-01-28


city
부산    16
서울    15
Name: count, dtype: int64

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [9]:
# 필터링 후엔 꼭 정렬해서 봐야 함. order by

expensive_products = (
    products
    .sort_values("price", ascending=False) # sort_values() # ascending(오름차순)=False이므로 내림차순()
    .head(10) # 10개를 볼것
)

display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
42,43,뷰티 상품 043,뷰티,197000
57,58,식품 상품 058,식품,197000
23,24,스포츠 상품 024,스포츠,196000
36,37,뷰티 상품 037,뷰티,193000
8,9,스포츠 상품 009,스포츠,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## 42. 작업용 복사본과 파생 컬럼

In [10]:
order_items_work = order_items.copy() # 카피본 생성, 원본 보존을 위해.

# line_total 열 추가
# 데이터 프레임에 새로운 컬럼을 추가할 때는 기존 데이터 프레임을 수정하지 않는다.
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

# 결과 확인(간단 빠름):
print(order_items_work.head())
# 결과 확인(예쁜 버전):
display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 43. 수작업 검증

In [11]:
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]

print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


## 44. 전체 주문상세 금액

In [12]:
all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255930000


현재 합계에는 취소 또는 환불 주문이 포함될 수 있으므로

완료 주문 기준 매출이 아니라 전체 주문상세 금액으로 표현한다.

## 45. 병합용 주문 컬럼 선택

In [13]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())

(301, 4)
   order_id  customer_id  order_date order_status
0         1          123  2025-09-08    completed
1         2           77  2026-04-28    cancelled
2         3          138  2026-02-02    cancelled
3         4           57  2026-05-12    cancelled
4         5          125  2026-04-17    cancelled


## 46. 주문상세와 주문 병합

In [14]:
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)
# 주문은 부모, 주문상세는 자식, 자식은 부모를 알고 있어야 한다.

## 47. 병합 검증

In [15]:
print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))

display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

# 정상 기준:
# • 행 수 유지
# • order_match가 모두 both

#미매칭 확인:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]

display(unmatched_orders.head())


병합 전 행 수: 766
병합 후 행 수: 766


order_match
both          765
left_only       1
right_only      0
Name: count, dtype: int64

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match
764,765,400,59,5,32000,160000,NaN,NaN,NaN,left_only


## 48. 완료 주문 분석셋

In [16]:
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)

print(

    "완료 주문 고객 수:",

    completed_sales["customer_id"].nunique(),

)

print(

    "완료 주문 매출:",

    completed_sales["line_total"].sum(),

)

order_status
completed    474
cancelled    163
refunded     128
NaN            1
Name: count, dtype: int64

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## 49. 필요한 상품 정보만 선택

In [20]:
products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()
# 상품 마스터의 price와 실제 주문 시점의 unit_price를 혼동하지 않도록 이번 병합에서는 price를 제외합니다.
print(products_for_merge)

    product_id product_name category
0            1  전자기기 상품 001     전자기기
1            2    도서 상품 002       도서
2            3  전자기기 상품 003     전자기기
3            4  생활용품 상품 004     생활용품
4            5    식품 상품 005       식품
..         ...          ...      ...
95          96  생활용품 상품 096     생활용품
96          97  전자기기 상품 097     전자기기
97          98   스포츠 상품 098      스포츠
98          99    뷰티 상품 099       뷰티
99         100    도서 상품 100       도서

[100 rows x 3 columns]


## 50. 완료 주문상세와 상품 병합

In [21]:
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 51. 카테고리별 매출

In [22]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [ ]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

#합계가 다르면 카테고리 결측, 상품 미매칭, 중복 병합과 필터 범위 차이를 확인합니다.

148990000
148990000
True


## 53. 상품별 매출

In [ ]:
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

# 판매량 상위와 매출 상위는 다를 수 있으므로 두 기준을 별도로 비교합니다.

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## 54. 주문 날짜 변환과 주문 월 생성

In [ ]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)


#order_month가 어떻게 들어갔을 까?
completed_items.info()
completed_items.head()

날짜 변환 실패: 0
<class 'pandas.DataFrame'>
RangeIndex: 474 entries, 0 to 473
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_item_id  474 non-null    int64         
 1   order_id       474 non-null    int64         
 2   product_id     474 non-null    int64         
 3   quantity       474 non-null    int64         
 4   unit_price     474 non-null    int64         
 5   line_total     474 non-null    int64         
 6   customer_id    474 non-null    float64       
 7   order_date     474 non-null    datetime64[us]
 8   order_status   474 non-null    str           
 9   order_match    474 non-null    category      
 10  product_name   474 non-null    str           
 11  category       474 non-null    str           
 12  product_match  474 non-null    category      
 13  order_month    474 non-null    string        
dtypes: category(2), datetime64[us](1), float64(1), int64(6), str(3), string(1

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match,product_name,category,product_match,order_month
0,1,1,100,3,102000,306000,123.0,2025-09-08,completed,both,도서 상품 100,도서,both,2025-09
1,2,1,87,5,25000,125000,123.0,2025-09-08,completed,both,도서 상품 087,도서,both,2025-09
2,3,1,7,3,142000,426000,123.0,2025-09-08,completed,both,도서 상품 007,도서,both,2025-09
3,4,1,9,3,193000,579000,123.0,2025-09-08,completed,both,스포츠 상품 009,스포츠,both,2025-09
4,13,6,83,3,24000,72000,87.0,2026-04-01,completed,both,전자기기 상품 083,전자기기,both,2026-04


## 55. 월별 매출

In [27]:
monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,7932000,8,8,74
1,2025-09,12922000,15,15,119
2,2025-10,12756000,18,16,135
3,2025-11,10573000,12,11,89
4,2025-12,5000000,8,8,51
5,2026-01,13111000,17,16,123
6,2026-02,16653000,19,18,137
7,2026-03,15848000,19,17,148
8,2026-04,11512000,15,15,136
9,2026-05,7657000,9,9,82


## 56. 고객별 구매 금액

In [29]:
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)

print(customer_sales)

    customer_id  total_sales  order_count  quantity_sold
0           3.0      3178000            2             26
1           4.0       603000            1              6
2           5.0      2004000            2             13
3           6.0      1349000            2             14
4           7.0      1173000            1              9
..          ...          ...          ...            ...
95        145.0      1032000            3             12
96        146.0      1000000            1              5
97        147.0      2990000            2             21
98        148.0       271000            1              2
99        149.0      1295000            1             10

[100 rows x 4 columns]


## 57. 고객 속성 연결
개인정보 최소화를 위해 이름은 제외합니다.

In [30]:
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()

customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))

customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117.0,4100000,5,48,F,65,성남,both
62,102.0,3996000,4,35,M,60,고양,both
51,83.0,3880000,4,39,F,22,수원,both
21,30.0,3590000,5,32,F,32,서울,both
29,40.0,3523000,4,27,M,23,서울,both
13,20.0,3191000,2,25,F,20,인천,both
0,3.0,3178000,2,26,F,61,성남,both
70,111.0,3153000,3,38,F,41,광주,both
42,66.0,3093000,4,30,F,39,서울,both
97,147.0,2990000,2,21,M,19,부산,both


## 58. 결과 폴더 생성

In [31]:
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

/Users/chaehuiju/dev/ai-data-analysis/reports/chapter04


## 59. 결과 파일 저장

In [32]:
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 301
product_sales.csv True 4734
monthly_sales.csv True 402
customer_sales.csv True 3547


## 60. 저장 결과 다시 읽기

In [ ]:
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

# 저장 후 다시 읽어 컬럼, 행 수와 한글 표시가 유지되는지 확인합니다.

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


## 61. 병합 점검 함수

In [34]:
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )

check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 766
병합 후 행 수: 766
order_match
both          765
left_only       1
right_only      0
Name: count, dtype: int64


## 62. 집계 합계 검증 함수

In [35]:
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


## 63. 안전한 LLM 요청 정보

LLM에는 다음 정보를 제공합니다.

• DataFrame 이름


• 한 행의 의미

• 실제 컬럼명

• dtype 요약

• 주요 키와 관계

• 실제 주문 상태값

• 분석 범위

• 원하는 결과 컬럼

• 검증 기준

고객 이름, 이메일, 전화번호, 주소, 원본 주문 행 전체와 API 키는 제공하지 않습니다.

64. pandas 코드 요청 프롬프트

 

나는 온라인 쇼핑몰 데이터를 pandas로 분석하고 있습니다.

분석 목표:

완료 주문 기준 카테고리별 매출을 계산합니다.

DataFrame과 한 행의 의미:

- orders: 주문 한 건

- order_items: 주문에 포함된 상품 한 항목

- products: 상품 한 개

실제 컬럼:

- orders:

  order_id, customer_id, order_date,

  payment_method, order_status

- order_items:

  order_item_id, order_id, product_id,

  quantity, unit_price

- products:

  product_id, product_name, category, price

주요 관계:

- order_items.order_id → orders.order_id

  many_to_one

- order_items.product_id → products.product_id

  many_to_one

분석 범위:

- order_status가 completed인 주문만 포함

- line_total = quantity × unit_price

- 주문 수는 order_id의 고유 개수

- 매출은 line_total 합계

원하는 결과:

- category

- total_sales

- order_count

- customer_count

- quantity_sold

검증 요구사항:

1. 각 merge에 validate를 사용해 주세요.

2. indicator로 미매칭을 확인해 주세요.

3. 병합 전후 행 수를 출력해 주세요.

4. 카테고리 합계와 완료 주문 전체 합계를 비교해 주세요.

5. 실제로 존재하지 않는 컬럼을 만들지 마세요.

6. 코드 실행 전 확인할 항목도 설명해 주세요.